# Calculus for Machine Learning

This notebook reviews the calculus concepts used throughout these notes: derivatives, integrals, gradients, the chain rule, backpropagation, and constrained optimization. Each section pairs a short mathematical explanation with a runnable Python example.

Run every cell from top to bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

np.random.seed(0)
plt.rcParams["figure.figsize"] = (9, 5)


## 1. The derivative

The derivative of a function $f$ at a point $x$ measures the instantaneous rate of change:

\(f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}\)

Geometrically, $f'(x)$ is the slope of the tangent line to the graph of $f$ at $x$. Below we plot $f(x)=x^2$ together with its derivative, first estimated numerically with the difference quotient and then from the exact rule $f'(x)=2x$.

In [ ]:
def f(x):
    return x ** 2

x = np.linspace(-4, 4, 400)
y = f(x)

# Numerical derivative with the difference quotient
h = 1e-4
y1 = (f(x + h) - f(x)) / h

# Exact derivative: f'(x) = 2x
y2 = 2 * x

fig, axes = plt.subplots(1, 2)

axes[0].plot(x, y, lw=2)
axes[0].set_title("$f(x) = x^2$")
axes[0].grid(alpha=0.3)

axes[1].plot(x, y1, lw=2, label="numerical difference quotient")
axes[1].plot(x, y2, "--", lw=2, label="exact $f'(x)=2x$")
axes[1].set_title("Derivative")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 2. The integral as accumulation

The integral accumulates a quantity over an interval. For a rate function $f(t)$, the total accumulated amount from $a$ to $b$ is

\(\int_a^b f(t) \, dt\)

Numerically, we approximate the integral by a Riemann sum: sample $f$ on a fine grid, multiply each value by the grid spacing $\Delta t$, and add them up. The Fundamental Theorem of Calculus connects the two ideas: integration undoes differentiation.

In [ ]:
def f(t):
    return 2 * t

delta_t = 0.1
t = np.arange(-4, 4, delta_t)

# Riemann sum: accumulate f(t) * delta_t
fx = f(t) * delta_t
accumulated = fx.cumsum()

# Comparison: accumulated should track t^2 (up to a constant)
plt.plot(t, accumulated, lw=2, label="Riemann sum of 2t")
plt.plot(t, t ** 2, "--", lw=2, label="$t^2$ (exact antiderivative)")
plt.axhline(0, color="black", lw=0.5)
plt.axvline(0, color="black", lw=0.5)
plt.title("Integration accumulates the rate of change")
plt.xlabel("$t$")
plt.ylabel("accumulated value")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 3. Gradients and directional derivatives

For a function of several variables $f(\mathbf{x})$, the \emph{gradient} collects all partial derivatives into a vector:

\(\nabla f(\mathbf{x}) = \begin{bmatrix} \partial f/\partial x_1 \\ \vdots \\ \partial f/\partial x_d \end{bmatrix}\)

The gradient points in the direction of steepest ascent, and its magnitude is the rate of change in that direction. The \emph{directional derivative} along a unit vector $\mathbf{u}$ is $\nabla f(\mathbf{x})^\top \mathbf{u}$. Below we visualize the gradient field of $f(x,y)=x^2+y^2$: arrows point outward, away from the minimum at the origin.

In [ ]:
x = np.linspace(-3, 3, 20)
y = np.linspace(-3, 3, 20)
X, Y = np.meshgrid(x, y)

# Gradient components
FX = 2 * X
FY = 2 * Y

# Arrow direction and magnitude
magnitude = np.sqrt(FX ** 2 + FY ** 2)
magnitude[magnitude == 0] = 1  # avoid division by zero at the origin
UX = FX / magnitude
UY = FY / magnitude

plt.quiver(X, Y, UX, UY, magnitude, cmap="viridis")
plt.scatter([0], [0], color="red", s=60, zorder=5, label="minimum at (0,0)")
plt.title("Gradient field of $f(x,y)=x^2+y^2$ (arrows point uphill)")
plt.xlabel("$x$")
plt.ylabel("$y$")
plt.legend()
plt.axis("equal")
plt.grid(alpha=0.3)
plt.show()


## 4. Directional derivative = maximum along the gradient

For a fixed step size, the rate of change of $f$ in direction $\mathbf{u}$ is $\nabla f \cdot \mathbf{u}$. By the Cauchy–Schwarz inequality this is maximized when $\mathbf{u}$ points along $\nabla f$ itself. We demonstrate this numerically for $f(x,y)=x^2+y^3$ at the point $(2,3)$: we scan all directions and confirm the maximum occurs along the gradient.

In [ ]:
def f(x, y):
    return x ** 2 + y ** 3

x0, y0 = 2.0, 3.0
step = 1e-4

# Exact gradient at (x0, y0): df/dx = 2x, df/dy = 3y^2
grad_exact = np.array([2 * x0, 3 * y0 ** 2])
print("Exact gradient at (2,3):", grad_exact)

# Scan all directions
angles = np.linspace(0, 2 * np.pi, 721)
rates = []
for theta in angles:
    u = np.array([np.cos(theta), np.sin(theta)])  # unit direction
    rate = (f(x0 + step * u[0], y0 + step * u[1]) - f(x0, y0)) / step
    rates.append(rate)
rates = np.array(rates)

best_idx = np.argmax(rates)
best_u = np.array([np.cos(angles[best_idx]), np.sin(angles[best_idx])])

print(f"Best direction from scan:  u = ({best_u[0]:.4f}, {best_u[1]:.4f})")
print(f"Gradient direction:          u = ({grad_exact[0]/np.linalg.norm(grad_exact):.4f}, {grad_exact[1]/np.linalg.norm(grad_exact):.4f})")
print(f"Maximum directional rate:    {rates[best_idx]:.6f} (exact ||grad|| = {np.linalg.norm(grad_exact):.6f})")


## 5. The chain rule and backpropagation

The chain rule is the engine of backpropagation. If $h(\mathbf{x}) = f(\mathbf{g}(\mathbf{x}))$, then

\(\nabla h(\mathbf{x}) = J_\mathbf{g}(\mathbf{x})^\top \, \nabla f(\mathbf{g}(\mathbf{x}))\)

where $J_\mathbf{g}$ is the Jacobian matrix of $\mathbf{g}$. In a neural network, this means gradients flow backward from the loss through each layer, multiplying by the local Jacobian at every step.

We illustrate with a vector-valued function $\mathbf{g}(x,y) = (2xy,\ x^2y)$ and compute its Jacobian symbolically with `sympy`.

In [ ]:
from sympy.abc import x, y
from sympy import Matrix, pprint

# g(x, y) = (2xy, x^2 y)
g = Matrix([2 * x * y, x ** 2 * y])
variables = Matrix([x, y])
J = g.jacobian(variables)

print("g(x, y) =")
pprint(g)
print("\nJacobian J_g(x, y) =")
pprint(J)

# Numerical check at (x, y) = (1.5, 2.0)
xv, yv = 1.5, 2.0
J_num = np.array([
    [2 * yv, 2 * xv],
    [2 * xv * yv, xv ** 2]
])
print("\nNumerical J at (1.5, 2.0):\n", J_num)


## 6. Activation functions and their derivatives

Neural networks apply nonlinear activations such as sigmoid and ReLU. During backpropagation we need their derivatives:

\(\sigma(z) = \frac{1}{1+e^{-z}}, \qquad \sigma'(z) = \sigma(z)(1-\sigma(z))\)

\(\operatorname{ReLU}(z) = \max(0,z), \qquad \operatorname{ReLU}'(z) = \mathbf{1}\{z > 0\}\)

Below we plot both functions and their derivatives.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def dsigmoid(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def drelu(z):
    return (z > 0).astype(float)

z = np.linspace(-6, 6, 400)

fig, axes = plt.subplots(1, 2)

axes[0].plot(z, sigmoid(z), lw=2, label="sigmoid")
axes[0].plot(z, relu(z), lw=2, label="ReLU")
axes[0].set_title("Activation functions")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(z, dsigmoid(z), lw=2, label="sigmoid'")
axes[1].plot(z, drelu(z), lw=2, label="ReLU'")
axes[1].set_title("Derivatives")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 7. Constrained optimization with Lagrange multipliers

Many ML problems minimize an objective subject to constraints. For equality constraints $g_j(\mathbf{w}) = 0$, we form the Lagrangian

\(\mathcal{L}(\mathbf{w}, \boldsymbol{\lambda}) = f(\mathbf{w}) - \sum_j \lambda_j g_j(\mathbf{w})\)

and solve $\nabla_{\mathbf{w}} \mathcal{L} = \mathbf{0}, \, g_j(\mathbf{w}) = 0$. For inequality constraints we use KKT conditions. The example below finds the point on the line $w_0 + 2w_1 + w_2 = 1$ closest to the origin (equivalently, minimizes $w_0^2+w_1^2+w_2^2$ subject to that linear constraint).

In [ ]:
def objective(w):
    return w[0] ** 2 + w[1] ** 2 + w[2] ** 2

# Equality constraint: w0 + 2*w1 + w2 = 1
def constraint_eq(w):
    return w[0] + 2 * w[1] + w[2] - 1

w0 = np.array([1.0, 1.0, 1.0])
bounds = [(-10, 10), (-10, 10), (-10, 10)]
constraints = [{"type": "eq", "fun": constraint_eq}]

result = minimize(objective, w0, method="SLSQP", bounds=bounds, constraints=constraints)

print("Optimal w:", np.round(result.x, 6))
print("Constraint value (should be 0):", constraint_eq(result.x))
print("Objective value:", result.fun)

# Analytic check: projection of origin onto the plane
normal = np.array([1.0, 2.0, 1.0])
w_exact = normal / (normal @ normal)  # = normal / ||normal||^2 since RHS = 1
print("Analytic solution:", np.round(w_exact, 6))


## Key takeaways

- **Derivative**: local slope / rate of change; estimated numerically with difference quotients.
- **Integral**: accumulation of a rate; the Riemann sum approximates it, and integration inverts differentiation.
- **Gradient**: vector of partial derivatives; points uphill, its norm is the steepest rate of change.
- **Chain rule**: gradients compose through Jacobians — this is backpropagation.
- **Activations**: sigmoid and ReLU (and their derivatives) are the building blocks of neural networks.
- **Constrained optimization**: Lagrange multipliers/KKT handle constraints; `scipy.optimize.minimize` solves them numerically.